# **Libraries needed**

In [7]:
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from huggingface_hub import notebook_login

notebook_login()

# **Load data**

In [ ]:
import os
import sys
if 'google.colab' in sys.modules:
    %cd /content/
    # remove local directory if it already exists
    if os.path.isdir("datavis"):
        !rm -rf {"datavis"}
    !git clone https://github.com/simon-mellergaard/datavis.git
    %cd /content/datavis/Data

In [ ]:

df = pd.read_excel(path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **TESTING**

In [9]:
COL_TITLE = "titel"
OUT_DIR = "/content/drive/MyDrive/DATA VIS TEST/"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = f"{OUT_DIR}/education_cluster_mapping.csv"
OUT_XLSX = f"{OUT_DIR}/education_cluster_mapping.xlsx"

df = df[df[COL_TITLE].astype(str).str.len() > 0].copy()

# Clusters
clusters = [
    ("Sundhed & Omsorg",
     "Uddannelser inden for sundhed, pleje, patientbehandling, terapi og omsorg."),
    ("Business, Økonomi & Ledelse",
     "Uddannelser om forretning, finans, marketing, økonomi og ledelse."),
    ("Ingeniør, IT & Data",
     "Uddannelser om ingeniørfag, software, data, informationsteknologi og produktion."),
    ("Samfund, Jura & Forvaltning",
     "Uddannelser om jura, politik, samfund, offentlig forvaltning og socialt arbejde."),
    ("Natur, Miljø & Fødevarer",
     "Uddannelser om naturvidenskab, miljø, landbrug, naturressourcer og fødevarer."),
    ("Uddannelse, Pædagogik & Social",
     "Uddannelser om undervisning, pædagogik, didaktik og sociale indsatser."),
    ("Kunst, Design, Medier & Kommunikation",
     "Uddannelser om kunst, arkitektur, design, medier, kommunikation og journalistik."),
]
cluster_labels = [c[0] for c in clusters]
label_texts = [f"{name}. {desc}" for name, desc in clusters]

# Model
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(MODEL_NAME, device=device)

label_emb = model.encode(
    label_texts,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True
)
titles = df[COL_TITLE].astype(str).str.strip().add(" uddannelse").tolist()
text_emb = model.encode(
    titles,
    batch_size=128,
    convert_to_tensor=True,
    normalize_embeddings=True
)

sims = text_emb @ label_emb.T  # [N, 7]
best_scores, best_idx = torch.max(sims, dim=1)

df["cluster_label"] = [cluster_labels[i] for i in best_idx.tolist()]
df["cluster_score"] = best_scores.detach().cpu().numpy()

# low-confidence bucket
THRESHOLD = 0.40
df.loc[df["cluster_score"] < THRESHOLD, "cluster_label"] = "Øvrige/Ukendt"

# Save
out = df[[COL_TITLE, "cluster_label", "cluster_score"]].copy()
out.to_csv(OUT_CSV, index=False)
out.to_excel(OUT_XLSX, index=False)

print(f"Saved CSV:   {OUT_CSV}")
print(f"Saved Excel: {OUT_XLSX}")
print("\nCounts per cluster:")
print(out["cluster_label"].value_counts())


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Saved CSV:   /content/drive/MyDrive/DATA VIS TEST//education_cluster_mapping.csv
Saved Excel: /content/drive/MyDrive/DATA VIS TEST//education_cluster_mapping.xlsx

Counts per cluster:
cluster_label
Ingeniør, IT & Data                      549
Uddannelse, Pædagogik & Social           533
Business, Økonomi & Ledelse              280
Kunst, Design, Medier & Kommunikation    228
Natur, Miljø & Fødevarer                 215
Sundhed & Omsorg                         211
Øvrige/Ukendt                             92
Samfund, Jura & Forvaltning               41
Name: count, dtype: int64
